In [2]:
import pandas as pd
import numpy as np

class Graph_struct:
   def __init__(self, V):
      self.V = V
      self.adj = [[] for i in range(V)]

   def DFS_Utililty(self, temp, v, visited):
      visited[v] = True
      temp.append(v)
      for i in self.adj[v]:
         if visited[i] == False:
            temp = self.DFS_Utililty(temp, i, visited)
      return temp

   def add_edge(self, v, w):
      self.adj[v].append(w)
      self.adj[w].append(v)

   def connected_components(self):
      visited = []
      conn_compnent = []
      for i in range(self.V):
         visited.append(False)
      for v in range(self.V):
         if visited[v] == False:
            temp = []
            conn_compnent.append(self.DFS_Utililty(temp, v, visited))
      return conn_compnent

df = pd.read_csv("Results/siamese/1.HAM10000/HAM10000_Siamese/similarity_matrix.csv")
# thresholds = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]
thresholds = [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6]

print("Method = Kendall")
for th in thresholds:
    print("Threshold = ",th)
    mat_sp = df.corr(method="kendall").round(2)
    tri_mat_sp = np.triu(mat_sp)
    spearman_matrix = mat_sp - tri_mat_sp
    ls=mat_sp.columns.tolist()
    ls_dict = {}
    for index, element in enumerate(ls):
        ls_dict[index] = element
    
    spearman_matrix_np = np.array(spearman_matrix)
    spearman_matrix_filtered = np.zeros((spearman_matrix.shape[0],spearman_matrix.shape[0]))

    for i in range(spearman_matrix_np.shape[0]):
        for j in range(spearman_matrix_np.shape[1]):
            if spearman_matrix_np[i,j] >= th:
                spearman_matrix_filtered[i,j] = 1
    
    my_instance = Graph_struct(54)
    for i in range(spearman_matrix_filtered.shape[0]):
        for j in range(spearman_matrix_filtered.shape[1]):
            if spearman_matrix_filtered[i,j] == 1:
                my_instance.add_edge(i, j)
    
    conn_comp = my_instance.connected_components()
    x=[]
    for i in range(len(conn_comp)):
        x.append(sorted(list(conn_comp[i])))
    
    sel_features = []
    rej_features = []
    for i in range(len(x)):
        sel_features.append(x[i][0])
        rej_features.append(x[i][1:])
    
    temp = list(filter(None, rej_features))
    rejected_features_flat = [x for xs in temp for x in xs]
    
    sel_features_names=[]
    rej_features_names=[]
    for k, v in ls_dict.items():
        if k in sel_features:
            sel_features_names.append(v)
        else:
            rej_features_names.append(v)
    
    print('Selected : ',len(sel_features)          , sel_features_names) 
    print('Rejected : ',len(rejected_features_flat), rej_features_names, "\n") 

Method = Kendall
Threshold =  0.1
Selected :  2 ['AutoContrast', 'Original']
Rejected :  52 ['BoxBlur', 'CLAHE', 'ChannelShuffle', 'ChromaticAberration', 'CoarseDropout', 'ColorJitter', 'Defocus', 'Dilation', 'ElasticTransform', 'Emboss', 'Equalize', 'EqualizeYUV', 'Erosion', 'Frost', 'Gamma', 'GaussNoise', 'GaussianBlur', 'GlassBlur', 'HorizontalFlip', 'HueSaturationValue', 'Invert', 'JPEGCompression', 'MedianBlur', 'MotionBlur', 'OpticalDistortion', 'Perspective', 'Posterize', 'RGBShift', 'RandomBrightnessContrast', 'RandomFog', 'RandomGravel', 'RandomGridShuffle', 'RandomRain', 'RandomShadow', 'RandomSnow', 'RandomSunFlare', 'RandomToneCurve', 'Rotate', 'SaltandPepper', 'Scale', 'Sharpen', 'ShearX', 'ShearY', 'ShiftScaleRotate', 'Solarize', 'Speckle', 'ToGray', 'ToSepia', 'Translate', 'VerticalFlip', 'Vignette', 'ZoomBlur'] 

Threshold =  0.15
Selected :  4 ['AutoContrast', 'ElasticTransform', 'Original', 'ZoomBlur']
Rejected :  50 ['BoxBlur', 'CLAHE', 'ChannelShuffle', 'ChromaticAb